# Notebook 07 — Benchmark and Evaluation

> **阶段**：Stage 4 Evaluate · **预计时间**：30–60 分钟 + 官方评测运行时间 · **平台**：Kaggle Notebook

| 资源 | 估算 | 说明 |
| --- | --- | --- |
| GPU VRAM | 不需要 GPU | 评测是 CPU 任务 |
| 磁盘 | 约 0.5 GB | 官方评测仓库 + 结果 |
| 实测时间 | 46 s（12 页 smoke 自检） | 官方评测时间待实测 |
| Internet | 需要（首次） | git clone 官方评测仓库 |


# Learning Objectives

完成本 Notebook 后，你应该能够：

- 使用锁定 commit 的 OmniDocBench 官方评测仓库；
- 把 baseline 的 DocTags 预测导出为官方评测输入（md2md / end2end）；
- 运行官方评测并汇总总体与分组（语言/文档类型/版面）结果；
- 区分「官方指标」与「非官方 smoke 自检」。


# Why This Matters

论文里报告的分数必须来自官方评测定义，而不是自造指标。同时，一个 Overall 分数会隐藏分组差异——分组切片才是后续错误分析的前提。


# Concepts

- **官方评测仓库**：`opendatalab/OmniDocBench` 无 release/tag，锁定 commit `193627ae…`；
- **两条端到端路径**：end2end（结构化 JSON）与 md2md（Markdown）；
- **官方指标族**：Edit Distance / BLEU / METEOR（文本）、TEDS（表格）、CDM（公式）、mAP（版面）、Reading Order 指标——以官方脚本实现为准；
- **红线**：本 Notebook 的 normalized edit distance 只是 pipeline 自检，**不是官方指标**，不能用于报告结论。


## Step 1 — 准备官方评测仓库（锁定 commit）

首次运行会 `git clone` 官方仓库并 checkout 到锁定 commit；再次运行校验 commit 一致。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import evaluation
from src.config import project_root

repo = evaluation.ensure_eval_repo(project_root() / 'third_party')
print('官方评测仓库:', repo)
print('锁定 commit:', evaluation.OFFICIAL_COMMIT)


## Step 2 — 从 baseline 缓存导出 Markdown 预测（md2md）

把 `results/baseline/predictions/*.json` 里的 DocTags 转成 Markdown 文件。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data, evaluation
from src.config import load_config, project_root

cfg = load_config()
data_root = data.find_dataset_root()
base = project_root() / cfg['paths']['baseline_dir']
md_dir = project_root() / cfg['paths']['benchmark_dir'] / 'md2md'
files = evaluation.export_markdown_predictions(base / 'predictions', data_root, md_dir)
print('导出 Markdown 预测:', len(files), '个 ->', md_dir)


## Step 3 — 运行官方评测

官方 CLI 命令模板在 `configs/default.yaml` 的 `omnidocbench_eval` 段。首次运行时按锁定 commit 的官方 README 填写模板后重跑本格；模板为空时自动跳过官方评测，只做非官方 smoke 自检。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import evaluation
from src.config import load_config, project_root

cfg = load_config()
bench = project_root() / cfg['paths']['benchmark_dir']
log = evaluation.run_official_eval(
    cfg, repo, gt_dir=data_root, pred_dir=md_dir,
    output_dir=bench, eval_kind='md2md',
)
if log is None:
    print('⚠️ 官方 CLI 模板未配置（configs/default.yaml），本次跳过官方评测；')
    print('⚠️ 下面是「非官方 smoke 自检」，仅验证 pipeline 贯通，不得作为成绩。')
else:
    print('官方评测日志:', log)


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data, evaluation

annotations = data.load_annotations(data_root)
rows = evaluation.sanity_check(base / 'predictions', annotations, data_root)
print('非官方 smoke 自检样本数:', len(rows))
print(rows[0])


## Step 4 — 汇总与分组切片

按 document_type / language / layout 分组看均值：一个 Overall 分数会隐藏的差异从这里开始显现。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
from src import data, evaluation
from src.config import project_root

summary = evaluation.build_summary_table(rows)
print(json.dumps(summary, ensure_ascii=False, indent=2))
bench = project_root() / cfg['paths']['benchmark_dir']
data.write_json(summary, bench / 'summary.json')
data.write_json(rows, bench / 'sanity_rows.json')


## Step 5 — Baseline / SFT / LoRA 对比表骨架

Phase 3 的 Notebook 05/06 会把 SFT 与 LoRA 结果填进同一张表。现在先把 Baseline 行放好，确保三个模型使用同一评测流程。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import pandas as pd

comparison = pd.DataFrame(columns=['model', 'text', 'table', 'formula', 'reading_order', 'overall', 'notes'])
# 官方评测出分后，从官方输出 JSON 填回；smoke 自检值只写进 notes，不写进指标列
comparison.loc[0] = ['baseline_smoldocling', None, None, None, None, None, 'smoke: %d pages' % len(rows)]
display(comparison)


# What You Should Observe

- 官方评测与 smoke 自检的输出结构完全不同：前者有指标族与配置，后者只有 pipeline 贯通证据；
- 分组切片里，difficult subset（equation_hard 等）与普通页面的差距通常很明显；
- 官方评测的配置（指标参数、对齐方式）必须与锁定 commit 一致，否则数字不可比。


# Research Checkpoint

> **「一个 Overall Score 会隐藏大量问题」——用你今天的分组结果举例说明：哪类页面/语言/版面被平均分掩盖了？这如何影响你选择下一步研究问题？**

**TODO：** 把答案写在 `results/nb07/research_checkpoint.md`。


# Exercises

1. **TODO：** 按锁定 commit 的官方 README 填写 `configs/default.yaml` 的评测命令模板，跑通一次真正的官方 md2md 评测，并记录官方指标 JSON 的位置；有余力再用 `src.evaluation.export_end2end_predictions` 导出 end2end 候选格式（字段与类别映射需按官方 README 核对修正）；
2. **TODO：** 挑出 smoke 自检中得分最低的 3 页，手动对比 prediction 与 GT，初步判断错误来自 OCR、表格还是阅读顺序（详细分类在 Notebook 08）；
3. **TODO：** 把 baseline 结果填入对比表后，检查：SFT/LoRA 未来填表时，哪些前提（prompt、采样、评测 commit）必须保持不变？


# Takeaways

- 官方评测锁定 commit，指标以官方实现为准；
- smoke 自检 ≠ 官方成绩，两者用途不同；
- 分组切片是错误分析与研究问题的最早信号。

**下一步**：Phase 3 — [Notebook 03](03_Prompt_Engineering.ipynb)（Prompt 实验）、[Notebook 05](05_SFT_Fundamentals.ipynb) 与 [Notebook 06](06_LoRA_Fine_Tuning.ipynb)（训练）。
